In [2]:
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
home_path = '/gws/ssde/j25a/duicv/yuansun/'

In [3]:
period_list = ['summer', 'winter'] 
factor_list = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

## FR-Capitole

In [4]:
start = datetime.fromisoformat('2004-02-20T00:30:00')
summer_end = datetime.fromisoformat('2004-06-27T00:00:00')  # just an example
winter_end = datetime.fromisoformat('2005-01-02T00:00:00')  # just an example

interval = timedelta(minutes=30)
summer_n_timesteps = int((summer_end - start) / interval)
winter_n_timesteps = int((winter_end - start) / interval)
print(summer_n_timesteps, winter_n_timesteps)

6143 15215


In [6]:
date_list = ['2004-06-27-01800', '2005-01-02-01800']
start_date_list = ['2004-06-27T00:00:00', '2005-01-02T00:00:00']
end_date_list = ['2004-07-04T00:00:00', '2005-01-09T00:00:00']
season_list = ['summer', 'winter']
var1 = ['LWup', 'Qh', 'Qle', 'Qtau']
var2 = ['FIRE_U', 'FSH_U', 'EFLX_LH_TOT_U', 'TAUX']
rename_dict = dict(zip(var2, var1))
GRIDNAME='FR-Capitole'
gridname='FR-Cap'
observation = f'{home_path}0_lcz_sp/UrbanPlumber/Urban-PLUMBER_FullCollection_v1/' + GRIDNAME + '/timeseries/'+ GRIDNAME + '_clean_observations_v1.nc'
ds_ob = xr.open_dataset(observation)
ds_traffic = xr.open_dataset(f'{home_path}0_urban_traffic/archive/{gridname}_traffic/lnd/hist/{gridname}_traffic.clm2.h0.2004-02-20-03600.nc')
for p, period in enumerate(period_list):
    ds_ob_sel = ds_ob.sel(time=slice(start_date_list[p], end_date_list[p]))
    df_obs = ds_ob_sel[var1].to_dataframe().reset_index()
    df_obs['case'] = 'obs'
    #df_obs = df_obs.iloc[:-1]
    df_obs=df_obs.iloc[1:]
    season = season_list[p]
    casename = f'{gridname}_{season}'
    date = date_list[p]
    all_dfs = [df_obs]
    ds_traffic_sel = ds_traffic.sel(time=slice(start_date_list[p], end_date_list[p]))
    df_traffic = ds_traffic_sel[var2].to_dataframe().reset_index()
    df_traffic['time'] = pd.to_datetime(df_traffic['time'])
    df_traffic['time'] = df_traffic['time'].dt.round('min').dt.ceil('min')
    df_traffic.drop(columns=['lndgrid'], inplace=True)
    df_traffic = df_traffic.rename(columns=rename_dict) 
    df_traffic['case'] = 'traffic'
    df_traffic['Qtau'] = -df_traffic['Qtau']
    df_traffic=df_traffic.iloc[1:]
    for factor in factor_list:
        ds_factor = xr.open_dataset(f'{home_path}0_urban_traffic/archive/sensitivity2/0_land_output/{casename}2/{casename}.clm2.h0.{date}_{factor}.nc')
        df_factor = ds_factor[var2].to_dataframe().reset_index()
        df_factor['time'] = pd.to_datetime(df_factor['time'])
        df_factor['time'] = df_factor['time'].dt.round('min').dt.ceil('min')
        df_factor.drop(columns=['lndgrid'], inplace=True)
        df_factor = df_factor.rename(columns=rename_dict) 
        df_factor['case'] = factor
        df_factor['Qtau'] = -df_factor['Qtau']
        all_dfs.append(df_factor)
    all_dfs.append(df_traffic)    
    df_combined = pd.concat(all_dfs, ignore_index=True)        
    df_combined.to_csv(f'./data_for_figure/{GRIDNAME}_{season}.csv', index=False)
    df_combined.head()     

## Taylor diagram metrics

In [7]:
case_list = factor_list + ['traffic']

In [8]:
GRIDNAME='FR-Capitole'
for season in season_list:
    df_combined = pd.read_csv(f'./data_for_figure/{GRIDNAME}_{season}.csv')
    df_obs = df_combined[df_combined['case'] == 'obs'].reset_index(drop=True)
    std_result = []
    coef_result = []
    for factor in case_list:
        df_factor = df_combined[df_combined['case'] == str(factor)].reset_index(drop=True)
        for var in var1:
            df_sim = df_factor[['time', var]].copy()
            df_sim = df_sim.rename(columns={var: 'sim'})
            df_sim['obs'] = df_obs[var]
            df_sim.loc[df_sim['obs'].isna(), ['sim']] = np.nan
            df_sim.dropna(subset=['obs', 'sim'], inplace=True)
            std = float(np.std(df_sim['sim']))
            sdev = std/float(np.std(df_sim['obs']))
            coef = float(xr.corr(xr.DataArray(df_sim['obs']), xr.DataArray(df_sim['sim'])).values)
            std_result.append(sdev)
            coef_result.append(coef)
    df = pd.DataFrame({'factor': np.repeat(case_list, len(var1)), 
                       'var': var1 * len(case_list), 
                       'sdev': std_result, 'coef': coef_result})  
    df.to_csv(f'./data_for_figure/{GRIDNAME}_{season}_std_coef.csv', index=False)      
    df 